<a href="https://www.kaggle.com/code/pavankumar960/predicting-ai-ml-salaries-an-end-to-end-ml?scriptVersionId=243365556" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# Project Setup

In [ ]:
#Basic Libs
import pandas as pd
import numpy as np

#Visualization Libs
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

#Model Training Libs
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression

import warnings
warnings.filterwarnings("ignore")

# Data Loading

In [ ]:
path = "/kaggle/input/global-ai-job-market-and-salary-trends-2025/ai_job_dataset.csv"

In [ ]:
df = pd.read_csv(path)

# Initial Data Overview & Quick Checks

In [ ]:
df.info()

**Observation**
- job_id not useful
- job_title can be target
- Salary_usd will be main target
- salary_currency not useful
- experience level - good cat feature
- employment type - good cat feature 
- company_location - might affect salary so, good cat feature
- company_size - not useful but can fact-check for No.of Job posts vs Employees in each company
- employee_residence - not useful but can predict availability of jobs outside country
- remote_ratio - useful, can fact check by comparing employee_residence and company_location
- required_skills - good cat feature with count
- education_required - good cat feature
- years_experience - related to experience_level, good num feature
- industry - average cat feature
- posting_date - useful for time series
- application_deadline - related to posting_date
- job_description_length - useful, if you use AI to analysis all the description
- benefits_score - can be target
- company_name - cat feature 

In [ ]:
df.head()

In [ ]:
df.nunique()

In [ ]:
columns_list = ['job_title','experience_level','employment_type','company_location',
                'company_size','employee_residence','remote_ratio','education_required',
                'years_experience','industry','company_name']

unique_values = {col: df[col].unique() for col in columns_list}

In [ ]:
unique_values

In [ ]:
df.describe()

# Data Cleaning & Preparation

## Handle Missing Values

In [ ]:
df.isnull().sum()

## Handle Duplicates

In [ ]:
df.duplicated().sum()

## Drop Unnecessary/Redundant Columns

In [ ]:
columns_to_drop = ['job_id', 'salary_currency']

df = df.drop(columns=columns_to_drop)

# Feature Engineering

In [ ]:
df['log_salary'] = np.log1p(df['salary_usd'])

In [ ]:
df['num_required_skills'] = df['required_skills'].apply(lambda x: len([skill.strip() for skill in x.split(',') if skill.strip()]))


# Exploratory Data Analysis (EDA) & Visualization

## Univariate Analysis

In [ ]:
numerical_cols = [
    'salary_usd','log_salary',
    'years_experience',
    'job_description_length',
    'benefits_score',
    'num_required_skills',
    'remote_ratio'
]

print("\nDistributions and Outliers for Numerical Features:")
for col in numerical_cols:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Histogram plot
    sns.histplot(df[col], kde=True, ax=axes[0])
    axes[0].set_title(f'Distribution of {col}')
    axes[0].set_xlabel(col)
    axes[0].set_ylabel('Frequency')

    # Box plot
    sns.boxplot(y=df[col], ax=axes[1])
    axes[1].set_title(f'Box Plot of {col}')
    axes[1].set_ylabel(col)

    plt.tight_layout()
    plt.show()


categorical_cols = [
    'experience_level',
    'employment_type',
    'company_size',
    'education_required',
    'job_title',
    'company_location',
    'employee_residence', 
    'industry', 
    'company_name' 
]

print("\nValue Counts for Categorical Features:")
for col in categorical_cols:
    plt.figure(figsize=(10, 6))
    sns.countplot(data=df, y=col, order=df[col].value_counts().index, palette='viridis')
    plt.title(f'Distribution of {col}')
    plt.xlabel('Count')
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()




In [ ]:
all_skills = ', '.join(df['required_skills'].dropna())

wordcloud = WordCloud(width=800, height=400, background_color='white', collocations=False).generate(all_skills)

plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Most Frequent AI Job Skills', fontsize=20)
plt.show()


## Bivariate Analysis

In [ ]:
print("\nScatter Plots: Numerical Features vs. Salary USD")
for col in numerical_cols:
    if col != 'salary_usd':
        plt.figure(figsize=(10, 6))
        sns.scatterplot(data=df, x=col, y='salary_usd', alpha=0.6, hue='experience_level', palette='coolwarm', size='years_experience', sizes=(20, 200))
        plt.title(f'{col} vs. Salary USD')
        plt.xlabel(col)
        plt.ylabel('Salary USD')
        plt.tight_layout()
        plt.show()

print("\nBox Plots: Categorical Features vs. Salary USD")
for col in categorical_cols:
    plt.figure(figsize=(12, 7))
    sns.boxplot(data=df, x=col, y='salary_usd', palette='pastel', order=df.groupby(col)['salary_usd'].median().sort_values(ascending=False).index)
    plt.title(f'Salary USD by {col}')
    plt.xlabel(col)
    plt.ylabel('Salary USD')
    plt.xticks(rotation=45, ha='right') 
    plt.tight_layout()
    plt.show()

print("\nBivariate Analysis: Remote Work Patterns")
plt.figure(figsize=(14, 8))
sns.countplot(data=df, y='employee_residence', hue='remote_ratio', palette='magma', order=df['employee_residence'].value_counts().index)
plt.title('Employee Residence by Remote Ratio')
plt.xlabel('Count')
plt.ylabel('Employee Residence')
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 8))
sns.countplot(data=df, y='company_location', hue='remote_ratio', palette='magma', order=df['company_location'].value_counts().index)
plt.title('Company Location by Remote Ratio')
plt.xlabel('Count')
plt.ylabel('Company Location')
plt.tight_layout()
plt.show()



**Observation**
- 80% salaries is between 50k and 200k so, its right-skewed salary distribution
- so Made log_salary for it
- outliers - Need Normalization or Log-Transform

## Correlation Matrix

In [ ]:
numerical_df = df[numerical_cols].copy()

correlation_matrix = numerical_df.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix of Numerical Features')
plt.show()


# Data Preprocessing (Encoding & Scaling for Modeling)

In [ ]:
columns_to_drop_before_encoding = [
    'required_skills', 
    'posting_date',   
    'application_deadline',
    'salary_usd'   
]

columns_to_drop_before_encoding = [col for col in columns_to_drop_before_encoding if col in df.columns]
df_cleaned_for_encoding = df.drop(columns=columns_to_drop_before_encoding)

print(f"Columns dropped from features for encoding: {columns_to_drop_before_encoding}")
print("DataFrame shape after initial cleaning for encoding:", df_cleaned_for_encoding.shape)


ordinal_features = {
    'experience_level': ['EN', 'MI', 'SE', 'EX'], # Entry, Mid, Senior, Executive
    'education_required': ['Associate', 'Bachelor', 'Master', 'PhD'],
    'company_size': ['S', 'M', 'L'] # Small, Medium, Large
}
ordinal_feature_names_list = list(ordinal_features.keys()) # List of column names for ColumnTransformer

nominal_features = [
    'job_title',
    'employment_type',
    'company_location',
    'employee_residence',
    'industry',
    'company_name',
    'remote_ratio'
]
if 'remote_ratio' in df_cleaned_for_encoding.columns:
    df_cleaned_for_encoding['remote_ratio'] = df_cleaned_for_encoding['remote_ratio'].astype(str)


numerical_features = [
    'years_experience',
    'job_description_length',
    'benefits_score',
    'num_required_skills',
]

numerical_features = [col for col in numerical_features if col in df_cleaned_for_encoding.columns]
nominal_features = [col for col in nominal_features if col in df_cleaned_for_encoding.columns]
ordinal_feature_names_list = [col for col in ordinal_feature_names_list if col in df_cleaned_for_encoding.columns]


preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='mean'), numerical_features),
        ('ord', OrdinalEncoder(categories=[ordinal_features[col] for col in ordinal_feature_names_list],
                               handle_unknown='use_encoded_value', unknown_value=-1),
         ordinal_feature_names_list),
        ('nom', OneHotEncoder(handle_unknown='ignore', sparse_output=False), nominal_features) # sparse_output=False for dense output
    ],
    remainder='drop'
)


X = df_cleaned_for_encoding 
y = df['log_salary']   


pipeline = Pipeline(steps=[('preprocessor', preprocessor)])


X_processed_array = pipeline.fit_transform(X)
numerical_output_features = numerical_features
ordinal_output_features = ordinal_feature_names_list

ohe_output_features = pipeline.named_steps['preprocessor'].named_transformers_['nom'].get_feature_names_out(nominal_features)
all_processed_feature_names = numerical_output_features + ordinal_output_features + list(ohe_output_features)

X_processed = pd.DataFrame(X_processed_array, columns=all_processed_feature_names)
print(X_processed.columns.tolist())


# Feature Selection

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)
print(f"Data split into training ({X_train.shape[0]} samples) and testing ({X_test.shape[0]} samples).")

selector_f = SelectKBest(f_regression, k=50) 
selector_f.fit(X_train, y_train)
f_scores_df = pd.DataFrame({'Feature': X_train.columns, 'F_Score': selector_f.scores_, 'P_Value': selector_f.pvalues_})
f_scores_df = f_scores_df.sort_values(by='F_Score', ascending=False)
print("\nTop 20 Features by F-Score:")
print(f_scores_df.head(20))


selector_mi = SelectKBest(mutual_info_regression, k=50) 
selector_mi.fit(X_train, y_train)
mi_scores_df = pd.DataFrame({'Feature': X_train.columns, 'MI_Score': selector_mi.scores_})
mi_scores_df = mi_scores_df.sort_values(by='MI_Score', ascending=False)
print("\nTop 20 Features by Mutual Information Score:")
print(mi_scores_df.head(20))

rf_model_for_importance = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model_for_importance.fit(X_train, y_train)
feature_importances_df = pd.DataFrame({'Feature': X_train.columns, 'Importance': rf_model_for_importance.feature_importances_})
feature_importances_df = feature_importances_df.sort_values(by='Importance', ascending=False)
print("\nTop 20 Features by RandomForest Importance:")
print(feature_importances_df.head(20))

N_FEATURES_TO_SELECT = 60 
selected_features = feature_importances_df['Feature'].head(N_FEATURES_TO_SELECT).tolist()
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]
print(f"\nSelected {N_FEATURES_TO_SELECT} features based on RandomForest importance.")
print("Shape of X_train_selected:", X_train_selected.shape)
print("Shape of X_test_selected:", X_test_selected.shape)
print("First 10 selected feature names:", selected_features[:10])

# Model Training & Evaluation

In [ ]:
final_model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1,
                                     max_depth=10, min_samples_leaf=5)
print("\nTraining RandomForestRegressor model...")
final_model.fit(X_train_selected, y_train)
print("Model training complete.")

y_pred_log = final_model.predict(X_test_selected)

mae_log = mean_absolute_error(y_test, y_pred_log)
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))
r2_log = r2_score(y_test, y_pred_log)

print(f"\n--- Model Evaluation (Log Scale) ---")
print(f"Mean Absolute Error (Log Salary): {mae_log:.4f}")
print(f"Root Mean Squared Error (Log Salary): {rmse_log:.4f}")
print(f"R-squared (Log Salary): {r2_log:.4f}")


y_test_actual = np.expm1(y_test)
y_pred_actual = np.expm1(y_pred_log)

mae_actual_usd = mean_absolute_error(y_test_actual, y_pred_actual)
rmse_actual_usd = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
r2_actual_usd = r2_score(y_test_actual, y_pred_actual)

print(f"\n--- Model Evaluation (Actual USD Salary Scale) ---")
print(f"Mean Absolute Error (Actual USD Salary): ${mae_actual_usd:,.2f}")
print(f"Root Mean Squared Error (Actual USD Salary): ${rmse_actual_usd:,.2f}")
print(f"R-squared (Actual USD Salary): {r2_actual_usd:.4f}")


# Conclusion

- This notebook demonstrates a complete end-to-end process for predicting AI/ML job salaries.
- The model achieves an average absolute error of **$17,505.91** on the actual USD salary scale.
- Key insights from EDA and feature selection can inform further model improvements or strategic decisions.


<p style="font-size:30px;">If you found this notebook helpful, please upvote it; it will motivate me to post more notebooks.</p>